In [ ]:
import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Label mappings
LABEL2ID = {"other": 0, "drone": 1}
ID2LABEL = {0: "other", 1: "drone"}


In [ ]:
# Configuration
MODEL_ID = "facebook/wav2vec2-base"
MAX_AUDIO_LENGTH_SEC = 0.5
SAMPLING_RATE = 16000
MAX_LENGTH = SAMPLING_RATE * MAX_AUDIO_LENGTH_SEC

# Initialize feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)


In [ ]:
def preprocess_audio(examples):
    """Preprocess audio batches: convert to mono, extract features."""
    audio_arrays = []
    
    for audio_data in examples['audio']:
        # Convert to numpy array
        arr = np.array(audio_data, dtype=np.float32)
        
        # Handle multi-channel: convert to mono by averaging
        if arr.ndim > 1:
            arr = np.mean(arr, axis=0)
        
        # Ensure 1D array
        arr = np.squeeze(arr)
        if arr.ndim == 0:
            arr = np.array([arr])
        elif arr.ndim > 1:
            arr = arr.flatten()
        
        # Ensure non-empty
        if len(arr) == 0:
            arr = np.zeros(1, dtype=np.float32)
        
        audio_arrays.append(arr)
    
    # Extract features with padding/truncation
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLING_RATE,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    inputs['label'] = examples['label']
    return inputs


In [16]:
# Load dataset
dataset = load_dataset("Hibou-Foundation/big_ds_4_raw_wav_balanced")

n = 1
dataset = DatasetDict({
    split: ds.select(range(int(n * len(ds))))
    for split, ds in dataset.items()
})

print(f"Dataset sizes: {[f'{k}: {len(v)}' for k, v in dataset.items()]}")

# Preprocess
encoded_dataset = dataset.map(
    preprocess_audio,
    remove_columns=["audio"],
    batched=True,
    batch_size=32,
    num_proc=20
)


Dataset sizes: ['train: 352116', 'val: 43580', 'test: 43478']


Map (num_proc=20): 100%|██████████| 43478/43478 [00:10<00:00, 4216.22 examples/s]


In [17]:
# Initialize model
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Freeze feature encoder for faster training
model.freeze_feature_encoder()


/home/pierre/Documents/Projects/PST4/AI/.venv/lib/python3.13/site-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
# Metrics computation
def compute_metrics(eval_pred):
    """Compute accuracy for validation."""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    accuracy = accuracy_score(eval_pred.label_ids, predictions)
    return {"accuracy": accuracy}

# Training arguments
training_args = TrainingArguments(
    output_dir="./drone_detector",
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    logging_steps=20,
    logging_first_step=True,
    save_steps=2000,
    eval_steps=2000,
    learning_rate=4e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    warmup_steps=500,
    fp16=torch.cuda.is_available(),
    report_to=[],
    dataloader_num_workers=22
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["val"],
    processing_class=feature_extractor,  # Fixed deprecation warning
    compute_metrics=compute_metrics,
)


In [ ]:
# Train model
trainer.train()

# Evaluate on test set
test_results = trainer.evaluate(encoded_dataset["test"])
print(f"\nTest Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")


Traceback (most recent call last):
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocessing/queues.py", line 268, in _feed
    send_bytes(obj)
    ~~~~~~~~~~^^^^^
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocessing/connection.py", line 200, in send_bytes
    self._send_bytes(m[offset:offset + size])
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocessing/connection.py", line 427, in _send_bytes
    self._send(header + buf)
    ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocessing/connection.py", line 384, in _send
    n = write(self._handle, buf)
OSError: [Errno 9] Bad file descriptor
Traceback (most recent call last):
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocessing/queues.py", line 257, in _feed
    reader_close()
    ~~~~~~~~~~~~^^
  File "/home/pierre/Applications/miniconda3/lib/python3.13/multiprocess

Step,Training Loss,Validation Loss


In [ ]:
# Get predictions and evaluate
predictions = trainer.predict(encoded_dataset["test"])
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=list(ID2LABEL.values()), digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=list(ID2LABEL.values()),
    yticklabels=list(ID2LABEL.values())
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Test Set Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
def preprocess_single(example):
    """Process a single example"""
    audio_item = example['audio']
    arr = audio_item['array']

    # Ensure it's a numpy array
    if not isinstance(arr, np.ndarray):
        arr = np.array(arr)

    # Handle multi-channel audio: convert to mono by averaging channels
    if arr.ndim > 1:
        arr = np.mean(arr, axis=0)

    # Ensure it's 1D (squeeze any extra dimensions)
    arr = np.squeeze(arr)

    # Ensure it's 1D array (not scalar)
    if arr.ndim == 0:
        arr = arr.reshape(1)
    elif arr.ndim > 1:
        # Flatten if still multi-dimensional
        arr = arr.flatten()

    # Feature extraction
    inputs = feature_extractor(
        arr,
        sampling_rate=16000,
        padding=True,
        truncation=True,
        max_length=16000 * 5
    )

    inputs['label'] = example['label']
    inputs["input_values"] = inputs["input_values"][0]
    return inputs

# Load additional test datasets
test_datasets = load_dataset("Hibou-Foundation/drone_all_test_datasets")
test_splits = {
    "drone_test": test_datasets["drone_test"],
    "drone_test_2": test_datasets["drone_test_2"],
    "mic_parabole": test_datasets["mic_parabole"],
    "parabole2": test_datasets["parabole2"]
}

# Preprocess and evaluate each test set
for name, ds in test_splits.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {name} ({len(ds)} samples)")
    print('='*60)
    
    encoded_ds = ds.map(
        preprocess_single,
        remove_columns=["audio"],
        batched=False,
        num_proc=1
    )
    
    preds = trainer.predict(encoded_ds)
    y_pred = np.argmax(preds.predictions, axis=1)
    y_true = preds.label_ids
    
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=list(ID2LABEL.values()), digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=list(ID2LABEL.values()),
        yticklabels=list(ID2LABEL.values())
    )
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    plt.show()
